# Análisis Exploratorio — Dataset Olist

Exploración del dataset de e-commerce brasileño Olist (~100.000 pedidos, 2016–2018).

**Objetivo:** entender la distribución de los datos antes de construir el pipeline ETL y el modelo ML.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Estilo general
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13

DATA = "../data/raw"

# Carga de tablas
orders    = pd.read_csv(f"{DATA}/olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"
])
customers  = pd.read_csv(f"{DATA}/olist_customers_dataset.csv")
items      = pd.read_csv(f"{DATA}/olist_order_items_dataset.csv")
products   = pd.read_csv(f"{DATA}/olist_products_dataset.csv")
reviews    = pd.read_csv(f"{DATA}/olist_order_reviews_dataset.csv")
payments   = pd.read_csv(f"{DATA}/olist_order_payments_dataset.csv")

print("Tablas cargadas:")
for name, df in [("orders", orders), ("customers", customers), ("items", items),
                 ("products", products), ("reviews", reviews), ("payments", payments)]:
    print(f"  {name:12s} → {df.shape[0]:>7,} filas  {df.shape[1]} columnas")

## 1. Visión general — valores nulos

In [ ]:
nulls = orders.isnull().sum()
nulls = nulls[nulls > 0].rename("nulos")
pct   = (nulls / len(orders) * 100).round(1).rename("%")
print("Nulos en orders:")
print(pd.concat([nulls, pct], axis=1).to_string())

## 2. Volumen de pedidos en el tiempo

In [ ]:
monthly = (
    orders
    .set_index("order_purchase_timestamp")
    .resample("ME")["order_id"]
    .count()
    .reset_index()
)
monthly.columns = ["month", "num_orders"]

fig, ax = plt.subplots()
ax.fill_between(monthly["month"], monthly["num_orders"], alpha=0.3)
ax.plot(monthly["month"], monthly["num_orders"], marker="o", markersize=4)
ax.set_title("Pedidos por mes")
ax.set_xlabel("")
ax.set_ylabel("Nº pedidos")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

## 3. Distribución geográfica — pedidos por estado

In [ ]:
state_orders = (
    orders
    .merge(customers[["customer_id", "customer_state"]], on="customer_id")
    .groupby("customer_state")["order_id"]
    .count()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 8))
state_orders.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Pedidos por estado")
ax.set_xlabel("Nº pedidos")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

## 4. Top 10 categorías por revenue

In [ ]:
cat_revenue = (
    items
    .merge(products[["product_id", "product_category_name"]], on="product_id")
    .groupby("product_category_name")["price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .sort_values()
)

fig, ax = plt.subplots()
cat_revenue.plot(kind="barh", ax=ax, color="coral")
ax.set_title("Top 10 categorías por revenue")
ax.set_xlabel("Revenue total (R$)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x/1e6:.1f}M"))
plt.tight_layout()
plt.show()

## 5. Distribución de puntuaciones de reviews

In [ ]:
score_counts = reviews["review_score"].value_counts().sort_index()
pct = (score_counts / score_counts.sum() * 100).round(1)

fig, ax = plt.subplots()
bars = ax.bar(score_counts.index, score_counts.values, color=["#d9534f","#e8a838","#f0e040","#7fbf7f","#2e8b57"])
ax.set_title("Distribución de puntuaciones de reviews")
ax.set_xlabel("Puntuación")
ax.set_ylabel("Nº reviews")
for bar, p in zip(bars, pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f"{p}%", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## 6. Retrasos en entregas y su impacto en la puntuación

In [ ]:
delivered = orders.dropna(subset=["order_delivered_customer_date"])
delivered = delivered.copy()
delivered["delay_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]
).dt.days
delivered["on_time"] = delivered["delay_days"] <= 0

df_plot = (
    delivered
    .merge(reviews[["order_id", "review_score"]], on="order_id", how="left")
    .dropna(subset=["review_score"])
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: % entregas a tiempo vs tardías
on_time_pct = delivered["on_time"].value_counts(normalize=True) * 100
axes[0].pie(
    on_time_pct,
    labels=["A tiempo", "Con retraso"],
    autopct="%1.1f%%",
    colors=["#2e8b57", "#d9534f"],
    startangle=90,
)
axes[0].set_title("% Entregas a tiempo vs con retraso")

# Derecha: puntuación media según si llegó a tiempo o tarde
avg_score = df_plot.groupby("on_time")["review_score"].mean()
axes[1].bar(
    ["Con retraso", "A tiempo"],
    [avg_score[False], avg_score[True]],
    color=["#d9534f", "#2e8b57"],
)
axes[1].set_title("Puntuación media según puntualidad de entrega")
axes[1].set_ylabel("Review score medio")
axes[1].set_ylim(1, 5)
for i, v in enumerate([avg_score[False], avg_score[True]]):
    axes[1].text(i, v + 0.05, f"{v:.2f}", ha="center", fontsize=12)

plt.tight_layout()
plt.show()

## 7. Métodos de pago

In [ ]:
pay_type = payments.groupby("payment_type")["payment_value"].sum().sort_values(ascending=False)

fig, ax = plt.subplots()
pay_type.plot(kind="bar", ax=ax, color="mediumpurple", rot=0)
ax.set_title("Revenue por método de pago")
ax.set_ylabel("Revenue total (R$)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x/1e6:.0f}M"))
plt.tight_layout()
plt.show()